# 🚀 Google Colab Multi-Link Downloader (Bypass 87GB Disk Limit)

Ứng dụng tải file đa nguồn tốc độ cao trên Google Colab sử dụng **JDownloader 2 (JD2) Headless** và Direct Streaming Engine:
- **Direct-to-Drive Streaming (Bỏ qua đĩa ảo Colab)**: Dữ liệu tải được stream trực tiếp vào Google Drive (`/content/drive/MyDrive/Downloads/`) theo từng luồng tuần tự (single chunk), **không lưu trữ trung gian** trên đĩa ảo Colab.
- **Tải file không giới hạn dung lượng**: Hỗ trợ tải các file dung lượng khổng lồ **> 87GB** mà không làm đầy đĩa cục bộ Colab (dung lượng đĩa cục bộ giữ nguyên không đổi).
- **Vô hiệu hóa Pre-allocation & Sparse Files**: Ngăn chặn tình trạng tạo file rỗng chiếm hết ổ đĩa ảo hoặc lỗi hệ thống file FUSE của Google Drive.
- **Tương thích Antigravity IDE**: Hỗ trợ cả 2 chế độ: giao diện trực quan (Interactive GUI) hoặc chạy trực tiếp bằng dòng lệnh / console HTML (Direct Mode).


In [ ]:
# [1] Mount Google Drive và cài đặt môi trường Java JRE + myjdapi
import os, sys

# 1. Kiểm tra và Mount Google Drive
if os.path.exists('/content/drive/MyDrive') or os.path.exists('/content/drive/My Drive'):
    print("✅ Google Drive đã được mount sẵn tại /content/drive/MyDrive")
else:
    try:
        from google.colab import drive
        print("📂 Đang mount Google Drive...")
        drive.mount('/content/drive', timeout_ms=30000)
        print("✅ Google Drive đã sẵn sàng!")
    except Exception as e:
        print(f"⚠️ Lưu ý mount Google Drive trên Antigravity IDE: {e}")
        print("👉 Bạn cũng có thể mở Command Palette (Ctrl+Shift+P) -> gõ 'Colab: Mount Google Drive to Server'")

# 2. Cài đặt các gói công cụ và thư viện cần thiết
!apt-get update -qq && apt-get install -y -qq default-jre aria2 megatools
!pip install -q --upgrade myjdapi ipywidgets requests


In [ ]:
# [2] Khởi chạy Giao diện Tải file Đa nguồn JDownloader 2 (Direct-to-Drive)

"""
Google Colab Multi-Link Downloader with JDownloader 2 Headless Engine
Author: Antigravity Assistant
Description: High-speed multi-source downloader for Google Colab powered by JDownloader 2 (JD2).
Supports 1000+ file hosting providers with an interactive ipywidgets GUI matching the
Ultimate Downloader layout. Downloads files and automatically transfers them to Google Drive Downloads.
"""

import os
import re
import sys
import time
import json
import shutil
import subprocess
import threading
from typing import List, Optional, Tuple, Dict, Any
from uuid import uuid4

# Check and import requests
try:
    import requests
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests"])
    import requests

# Check and import myjdapi
try:
    import myjdapi
    from myjdapi import Myjdapi
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "myjdapi"])
    import myjdapi
    from myjdapi import Myjdapi

# Check and import ipywidgets & IPython
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML

# Detect Google Colab environment
IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

# Configure environment paths
SAMPLE_DOWNLOAD_URL = "https://disk.yandex.com/d/7OcBsRWfzoTozg"

if IS_COLAB:
    COLAB_ROOT = "/content"
    JD_INSTALL_DIR = os.path.join(COLAB_ROOT, "jdownloader")
    TEMP_DOWNLOAD_DIR = os.path.join(COLAB_ROOT, "temp_downloads")
    DRIVE_MOUNT_POINT = os.path.join(COLAB_ROOT, "drive")
    DRIVE_BASE = os.path.join(DRIVE_MOUNT_POINT, "MyDrive")
    if not os.path.exists(DRIVE_BASE) and os.path.exists(os.path.join(DRIVE_MOUNT_POINT, "My Drive")):
        DRIVE_BASE = os.path.join(DRIVE_MOUNT_POINT, "My Drive")
    DRIVE_DOWNLOADS = os.path.join(DRIVE_BASE, "Downloads")
else:
    COLAB_ROOT = os.path.abspath("./colab_env")
    JD_INSTALL_DIR = os.path.join(COLAB_ROOT, "jdownloader")
    TEMP_DOWNLOAD_DIR = os.path.join(COLAB_ROOT, "temp_downloads")
    DRIVE_BASE = os.path.join(COLAB_ROOT, "drive", "MyDrive")
    DRIVE_DOWNLOADS = os.path.join(DRIVE_BASE, "Downloads")

# Persistent configuration directory in Google Drive (survives Colab resets)
DRIVE_JD_DIR = os.path.join(DRIVE_BASE, "JDownloader")
DRIVE_JD_CFG = os.path.join(DRIVE_JD_DIR, "cfg")
DRIVE_MYJD_SETTINGS = os.path.join(DRIVE_JD_CFG, "org.jdownloader.api.myjdownloader.MyJDownloaderSettings.json")
DRIVE_CRED_BACKUP = os.path.join(DRIVE_JD_DIR, "myjd_credentials.json")

os.makedirs(TEMP_DOWNLOAD_DIR, exist_ok=True)
os.makedirs(JD_INSTALL_DIR, exist_ok=True)
os.makedirs(DRIVE_DOWNLOADS, exist_ok=True)
try:
    os.makedirs(DRIVE_JD_CFG, exist_ok=True)
except Exception:
    pass


def save_credentials_to_drive(email: str, password: str, device_name: str = "Colab_Downloader") -> bool:
    """Save MyJDownloader credentials permanently to Google Drive and local JDownloader cfg."""
    data = {
        "email": email.strip(),
        "password": password.strip(),
        "devicename": device_name.strip(),
        "autoconnectenabledv2": True,
        "directconnectmode": "LAN_WAN_MANUAL"
    }
    backup_data = {
        "email": email.strip(),
        "password": password.strip(),
        "devicename": device_name.strip(),
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")
    }

    # 1. Save to Google Drive persistent path
    try:
        os.makedirs(DRIVE_JD_CFG, exist_ok=True)
        with open(DRIVE_MYJD_SETTINGS, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        with open(DRIVE_CRED_BACKUP, "w", encoding="utf-8") as f:
            json.dump(backup_data, f, indent=2)
    except Exception as e:
        safe_print(f"⚠️ Note saving credentials to Drive: {e}")

    # 2. Save to local JDownloader installation cfg
    try:
        local_cfg = os.path.join(JD_INSTALL_DIR, "cfg")
        os.makedirs(local_cfg, exist_ok=True)
        with open(os.path.join(local_cfg, "org.jdownloader.api.myjdownloader.MyJDownloaderSettings.json"), "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
    except Exception as e:
        safe_print(f"⚠️ Note saving credentials locally: {e}")

    return True


def load_credentials_from_drive() -> Dict[str, str]:
    """Load saved credentials from Google Drive (if available)."""
    for p in [DRIVE_CRED_BACKUP, DRIVE_MYJD_SETTINGS]:
        if os.path.exists(p):
            try:
                with open(p, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    if data.get("email") and data.get("password"):
                        return {
                            "email": str(data.get("email", "")).strip(),
                            "password": str(data.get("password", "")).strip(),
                            "devicename": str(data.get("devicename", "Colab_Downloader")).strip()
                        }
            except Exception:
                pass
    return {"email": "", "password": "", "devicename": "Colab_Downloader"}


# --- UTILITY HELPERS ---
def format_size(bytes_val: int) -> str:
    """Format bytes into human-readable string."""
    if not bytes_val or bytes_val <= 0:
        return "Unknown size"
    val = float(bytes_val)
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if val < 1024.0:
            return f"{val:.1f} {unit}"
        val /= 1024.0
    return f"{val:.1f} PB"


def sanitize_filename(filename: str) -> str:
    """Sanitize filename to prevent invalid characters across filesystems."""
    if not filename:
        return f"download_{int(time.time())}"
    cleaned = re.sub(r'[\\/*?:"<>|]', "_", filename).strip()
    return cleaned if cleaned else f"download_{int(time.time())}"


def safe_print(msg: str, end: str = "\n"):
    """Safely print strings with emojis on Windows and Linux consoles without charmap encode errors."""
    try:
        print(msg, end=end, flush=True)
    except (UnicodeEncodeError, UnicodeError):
        try:
            print(msg.encode('ascii', errors='backslashreplace').decode('ascii'), end=end, flush=True)
        except Exception:
            pass


def get_colab_disk_info() -> Dict[str, Any]:
    """Get current Colab local root disk usage in real time."""
    try:
        total, used, free = shutil.disk_usage('/')
        return {
            "total_gb": round(total / (1024 ** 3), 2),
            "used_gb": round(used / (1024 ** 3), 2),
            "free_gb": round(free / (1024 ** 3), 2)
        }
    except Exception:
        return {"total_gb": 0.0, "used_gb": 0.0, "free_gb": 0.0}


# --- JDOWNLOADER 2 HEADLESS SERVICE MANAGER ---
class JD2Service:
    """Manages installation, configuration, daemon process, and connection for JDownloader 2."""

    def __init__(self, jd_dir: str = JD_INSTALL_DIR, port: int = 3128):
        self.jd_dir = jd_dir
        self.port = port
        self.cfg_dir = os.path.join(jd_dir, "cfg")
        self.jar_path = os.path.join(jd_dir, "JDownloader.jar")
        self.jd_api = Myjdapi()
        self.jd_api.set_app_key("ColabDownloaderApp")
        self.device = None
        self.process: Optional[subprocess.Popen] = None
        self._connected = False

    def is_installed(self) -> bool:
        return os.path.exists(self.jar_path) and os.path.getsize(self.jar_path) > 1024 * 100

    def install_prerequisites(self, status_callback: Optional[callable] = None):
        """Ensure Java JRE is installed on system."""
        if shutil.which("java") is None and IS_COLAB:
            if status_callback:
                status_callback("📦 Installing Java JRE (default-jre) for JDownloader 2...")
            try:
                subprocess.run(["apt-get", "update", "-qq"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
                subprocess.run(["apt-get", "install", "-y", "-qq", "default-jre"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            except Exception as e:
                print(f"⚠️ Warning installing Java: {e}")

    def download_jdownloader_jar(self, status_callback: Optional[callable] = None):
        """Download official JDownloader.jar if missing."""
        if not self.is_installed():
            if status_callback:
                status_callback("⬇️ Downloading JDownloader.jar...")
            url = "http://installer.jdownloader.org/JDownloader.jar"
            try:
                resp = requests.get(url, timeout=30, stream=True)
                resp.raise_for_status()
                with open(self.jar_path, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
            except Exception as e:
                raise RuntimeError(f"Failed to download JDownloader.jar: {e}")

    def configure_settings(
        self,
        email: str = "",
        password: str = "",
        device_name: str = "Colab_Downloader",
        download_folder: Optional[str] = None
    ):
        """
        Write configuration JSON files for Local API, Cloud Remote API, and Google Drive direct streaming.
        Tuned specifically for Google Drive FUSE:
        - Disable pre-allocation & sparse files to avoid FUSE write errors.
        - Set chunks = 1 (single connection) for sequential byte streaming.
        - Point default download folder directly to Google Drive Downloads.
        """
        os.makedirs(self.cfg_dir, exist_ok=True)

        target_download_folder = download_folder or DRIVE_DOWNLOADS
        os.makedirs(target_download_folder, exist_ok=True)

        # 1. Enable Local RemoteAPI on port 3128
        remote_api_cfg = os.path.join(self.cfg_dir, "org.jdownloader.api.RemoteAPIConfig.json")
        remote_api_data = {
            "deprecatedapienabled": True,
            "deprecatedapilocalhostonly": True
        }
        with open(remote_api_cfg, "w", encoding="utf-8") as f:
            json.dump(remote_api_data, f, indent=2)

        # 2. Configure General Download Settings for Google Drive FUSE streaming
        general_cfg = os.path.join(self.cfg_dir, "org.jdownloader.settings.GeneralSettings.json")
        general_data = {
            "defaultdownloadfolder": target_download_folder,
            "autoextractenabled": False,
            # Mandatory for Google Drive FUSE: Single chunk / sequential stream
            "maxsimultanousdownloads": 1,
            "maxchunksperserver": 1,
            "maxchunksperdownload": 1,
            # Mandatory for Google Drive FUSE: Disable sparse file / pre-allocation
            "createpreallocatedlargefiles": False,
            "createpreallocatedlargefilesenabled": False,
            "useallocatesparsefile": False,
            "allocatesparsefiles": False,
            "flushbufferlevel": 100,
            "maxbuffersize": 500
        }
        with open(general_cfg, "w", encoding="utf-8") as f:
            json.dump(general_data, f, indent=2)

        # 3. MyJDownloader Cloud Settings for Remote Web & App Management
        if email and password:
            myjd_cfg = os.path.join(self.cfg_dir, "org.jdownloader.api.myjdownloader.MyJDownloaderSettings.json")
            myjd_data = {
                "email": email.strip(),
                "password": password.strip(),
                "devicename": device_name.strip(),
                "autoconnectenabledv2": True,
                "directconnectmode": "LAN_WAN_MANUAL"
            }
            with open(myjd_cfg, "w", encoding="utf-8") as f:
                json.dump(myjd_data, f, indent=2)

    def restart_service_with_cloud(self, email: str, password: str, device_name: str = "Colab_Downloader", status_callback: Optional[callable] = None) -> bool:
        """Configure credentials and start/restart JDownloader 2 daemon to connect to MyJDownloader cloud."""
        self.configure_settings(email=email, password=password, device_name=device_name)

        # Terminate any existing JD2 process so it restarts with new MyJDownloader credentials
        if self.process:
            try:
                self.process.terminate()
                self.process.wait(timeout=3)
            except Exception:
                try:
                    self.process.kill()
                except Exception:
                    pass
            self.process = None

        if IS_COLAB:
            try:
                subprocess.run(["pkill", "-f", "JDownloader.jar"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            except Exception:
                pass

        self.install_prerequisites(status_callback)
        self.download_jdownloader_jar(status_callback)

        if shutil.which("java") is None:
            if not IS_COLAB:
                return True
            raise RuntimeError("Java is required to run JDownloader 2.")

        if status_callback:
            status_callback("🚀 Đang khởi động JDownloader 2 và đồng bộ thiết bị với my.jdownloader.org...")

        log_path = os.path.join(self.jd_dir, "jd2_service.log")
        cmd = ["java", "-Djava.awt.headless=true", "-jar", self.jar_path]
        try:
            self._log_file = open(log_path, "a", encoding="utf-8")
            self.process = subprocess.Popen(
                cmd,
                cwd=self.jd_dir,
                stdin=subprocess.DEVNULL,
                stdout=self._log_file,
                stderr=subprocess.STDOUT
            )
        except Exception:
            self.process = subprocess.Popen(
                cmd,
                cwd=self.jd_dir,
                stdin=subprocess.DEVNULL,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )

        return True

    def start_service(self, status_callback: Optional[callable] = None):
        """Start the JDownloader 2 process in background if not already active."""
        # Check if already responding
        try:
            if self.jd_api.direct_connect("127.0.0.1", self.port):
                self.device = self.jd_api.get_device()
                self._connected = True
                if status_callback:
                    status_callback("✅ Connected to running JDownloader 2 instance.")
                return True
        except Exception:
            pass

        if shutil.which("java") is None:
            if not IS_COLAB:
                # Running in local test environment without java
                return False
            raise RuntimeError("Java is required to run JDownloader 2.")

        if status_callback:
            status_callback("🚀 Starting JDownloader 2 Headless Service (this may take ~20s on first start)...")

        log_path = os.path.join(self.jd_dir, "jd2_service.log")
        cmd = ["java", "-Djava.awt.headless=true", "-jar", self.jar_path]
        try:
            self._log_file = open(log_path, "a", encoding="utf-8")
            self.process = subprocess.Popen(
                cmd,
                cwd=self.jd_dir,
                stdin=subprocess.DEVNULL,
                stdout=self._log_file,
                stderr=subprocess.STDOUT
            )
        except Exception:
            self.process = subprocess.Popen(
                cmd,
                cwd=self.jd_dir,
                stdin=subprocess.DEVNULL,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )

        # Poll connection to Local API
        start_time = time.time()
        timeout = 60
        while time.time() - start_time < timeout:
            time.sleep(2)
            try:
                if self.jd_api.direct_connect("127.0.0.1", self.port):
                    self.device = self.jd_api.get_device()
                    self._connected = True
                    if status_callback:
                        status_callback("✅ JDownloader 2 connected successfully.")
                    return True
            except Exception:
                pass

        if status_callback:
            status_callback("⚠️ Could not connect to Local API within timeout.")
        return False

    def get_service_logs(self, max_lines: int = 15) -> str:
        """Get recent log output from running JDownloader 2 service."""
        log_path = os.path.join(self.jd_dir, "jd2_service.log")
        if not os.path.exists(log_path):
            return "No log file found yet."
        try:
            with open(log_path, "r", encoding="utf-8", errors="replace") as f:
                lines = f.readlines()
                return "".join(lines[-max_lines:]) if lines else "Log is empty."
        except Exception as e:
            return f"Error reading log: {e}"

    def connect_myjd(self, email: str, password: str, device_name: str = "Colab_Downloader") -> bool:
        """Connect via MyJDownloader Cloud, retrying until device registers."""
        try:
            self.jd_api.connect(email, password)
            start_t = time.time()
            while time.time() - start_t < 20:
                try:
                    self.device = self.jd_api.get_device(device_name)
                    if self.device:
                        self._connected = True
                        return True
                except Exception:
                    pass
                time.sleep(2)
            self._connected = True
            return True
        except Exception as e:
            print(f"⚠️ MyJDownloader cloud connection error: {e}")
            return False


# --- GOOGLE DRIVE FILE TRANSFER ---
def transfer_to_google_drive(
    local_path: str,
    drive_folder: str = DRIVE_DOWNLOADS,
    progress_callback: Optional[callable] = None
) -> Tuple[bool, str]:
    """
    Safely transfers a completed file from Colab local disk to Google Drive.
    Uses buffered chunked transfer to avoid memory overhead and deletes local file to save disk space.
    """
    if not os.path.exists(local_path):
        return False, f"Source file does not exist: {local_path}"

    os.makedirs(drive_folder, exist_ok=True)
    filename = os.path.basename(local_path)
    dest_path = os.path.join(drive_folder, filename)

    if os.path.exists(dest_path):
        base, ext = os.path.splitext(filename)
        dest_path = os.path.join(drive_folder, f"{base}_{int(time.time())}{ext}")

    file_size = os.path.getsize(local_path)
    copied_bytes = 0
    buffer_size = 16 * 1024 * 1024  # 16 MB

    if progress_callback:
        progress_callback(0, f"Transferring to Google Drive: {os.path.basename(dest_path)}")

    try:
        with open(local_path, "rb") as fsrc, open(dest_path, "wb") as fdst:
            while True:
                chunk = fsrc.read(buffer_size)
                if not chunk:
                    break
                fdst.write(chunk)
                copied_bytes += len(chunk)
                if progress_callback and file_size > 0:
                    pct = (copied_bytes / file_size) * 100.0
                    progress_callback(pct, f"Moving to Drive: {pct:.1f}% ({format_size(copied_bytes)} / {format_size(file_size)})")

        if os.path.exists(dest_path) and os.path.getsize(dest_path) == file_size:
            os.remove(local_path)  # Delete local copy to free Colab disk
            return True, dest_path
        else:
            return False, "File size mismatch after moving to Google Drive"
    except Exception as e:
        return False, f"Transfer error: {str(e)}"


# --- IPYWIDGETS GUI APPLICATION ---
class ColabDownloaderApp:
    """Builds and manages the interactive UI matching the user's screenshots with JDownloader 2 backend."""

    def __init__(self, jd_service: Optional[JD2Service] = None):
        self.jd = jd_service or JD2Service()
        self.pending_items: List[Dict[str, Any]] = []
        self.sort_ascending = True
        self.is_downloading = False

        # Custom CSS for Dark Jupyter / Colab theme
        self.css_styles = widgets.HTML("""
        <style>
            .colab-dl-container {
                background-color: #1e1e1e;
                color: #e0e0e0;
                padding: 16px;
                border-radius: 8px;
                font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
            }
            .widget-textarea textarea {
                background-color: #2d2d2d !important;
                color: #f1f1f1 !important;
                border: 1px solid #444 !important;
                border-radius: 4px !important;
                font-family: monospace !important;
                font-size: 13px !important;
            }
            .widget-textarea textarea::placeholder {
                color: #888 !important;
            }
            .widget-select-multiple select {
                background-color: #252526 !important;
                color: #d4d4d4 !important;
                border: 1px solid #3c3c3c !important;
                font-family: monospace !important;
                font-size: 13px !important;
            }
            .widget-select-multiple select option:checked {
                background-color: #04395e !important;
                color: #ffffff !important;
            }
            .queue-header {
                font-size: 14px;
                font-weight: bold;
                color: #f0f0f0;
                margin-bottom: 6px;
            }
        </style>
        """)

        # 1. Links Textarea (Image 1)
        self.text_area = widgets.Textarea(
            description='Links:',
            placeholder='Paste Links Here (e.g., https://disk.yandex.com/d/...., Rapidgator, Mega)...',
            layout=widgets.Layout(width='98%', height='140px')
        )

        # 2. Action Buttons (Image 2)
        self.btn_resolve = widgets.Button(
            description="Resolve Links",
            button_style='success',
            icon='search',
            tooltip='Resolve links with JDownloader 2 and show Queue Preview',
            layout=widgets.Layout(width='140px', height='36px')
        )
        self.btn_quick = widgets.Button(
            description="Quick Download",
            button_style='primary',
            icon='bolt',
            tooltip='Automatically resolve and stream directly to Google Drive',
            layout=widgets.Layout(width='155px', height='36px')
        )
        self.btn_sample = widgets.Button(
            description="Sample Link",
            button_style='info',
            icon='link',
            tooltip='Insert Yandex Disk sample link (TOGETHER.BnB, ~9.5GB)',
            layout=widgets.Layout(width='130px', height='36px')
        )

        # 3. Queue Preview Container & Controls (Image 3)
        self.queue_header = widgets.HTML(
            "<div class='queue-header'>📄 <b>Queue Preview</b> <span style='font-weight:normal; color:#aaa;'>(Select items to manage)</span></div>"
        )
        self.queue_list = widgets.SelectMultiple(
            description='Queue:',
            options=[],
            layout=widgets.Layout(width='98%', height='160px')
        )

        # Toolbar Buttons (Under Queue Preview)
        self.btn_up = widgets.Button(description="▲ Up", layout=widgets.Layout(width='80px'))
        self.btn_down = widgets.Button(description="▼ Down", layout=widgets.Layout(width='80px'))
        self.btn_sort = widgets.Button(description="Sort A-Z", layout=widgets.Layout(width='90px'))
        self.btn_select_all = widgets.Button(description="Select All", layout=widgets.Layout(width='85px'))
        self.btn_none = widgets.Button(description="None", layout=widgets.Layout(width='65px'))
        self.btn_start = widgets.Button(
            description="▶ Start Download",
            button_style='success',
            layout=widgets.Layout(width='140px')
        )
        self.btn_cancel = widgets.Button(
            description="Cancel",
            button_style='warning',
            layout=widgets.Layout(width='75px')
        )
        self.btn_remove = widgets.Button(
            description="Remove",
            button_style='danger',
            layout=widgets.Layout(width='80px')
        )

        self.queue_toolbar = widgets.HBox([
            self.btn_up,
            self.btn_down,
            self.btn_sort,
            self.btn_select_all,
            self.btn_none,
            self.btn_start,
            self.btn_cancel,
            self.btn_remove
        ], layout=widgets.Layout(margin='8px 0 0 0'))

        self.queue_container = widgets.VBox([
            self.queue_header,
            self.queue_list,
            self.queue_toolbar
        ], layout=widgets.Layout(display='none', margin='12px 0'))

        # 4. Progress and Status Elements
        self.progress_bar = widgets.FloatProgress(
            value=0.0,
            min=0.0,
            max=100.0,
            description='Idle',
            bar_style='info',
            layout=widgets.Layout(width='98%', margin='8px 0')
        )
        self.status_label = widgets.HTML(value="")

        # Real-time Colab Disk Usage Widget (Requirement #2)
        self.disk_info_widget = widgets.HTML(value=self._get_disk_html())

        # Auto-load saved credentials from Google Drive
        saved_creds = load_credentials_from_drive()
        saved_email = saved_creds.get("email", "")
        saved_pwd = saved_creds.get("password", "")

        # Optional MyJDownloader Credentials Accordion (collapsed by default)
        self.myjd_email = widgets.Text(
            description='Email:',
            value=saved_email,
            placeholder='my.jdownloader.org email',
            layout=widgets.Layout(width='260px')
        )
        self.myjd_pass = widgets.Password(
            description='Password:',
            value=saved_pwd,
            placeholder='Password',
            layout=widgets.Layout(width='220px')
        )
        self.btn_login = widgets.Button(
            description='Login & Lưu Drive' if not saved_email else 'Login lại (Drive)',
            button_style='success' if saved_email else 'primary',
            icon='check' if saved_email else 'sign-in',
            tooltip='Đăng nhập và tự động lưu thông tin vào Google Drive (không mất khi reset Colab)',
            layout=widgets.Layout(width='160px')
        )
        self.btn_login.on_click(self._on_login_myjd)

        self.btn_view_log = widgets.Button(
            description='Xem Log JD2',
            button_style='info',
            icon='file-text',
            tooltip='Xem log tiến trình JDownloader 2 thời gian thực',
            layout=widgets.Layout(width='120px')
        )
        self.btn_view_log.on_click(self._on_view_log)

        cred_info_html = """
        <div style='font-size:12px; color:#ccc; line-height:1.5; margin-bottom:8px;'>
            💾 <b>Tự động lưu vào Google Drive:</b> Thông tin đăng nhập được lưu tại <code>MyDrive/JDownloader/cfg/</code>, không bị mất khi reset runtime.<br>
            🌐 <b>Quản lý từ xa:</b> Sau khi đăng nhập, mở <a href='https://my.jdownloader.org' target='_blank' style='color:#00e676; font-weight:bold;'>my.jdownloader.org</a> hoặc App điện thoại để quản lý & giải captcha.<br>
            ⚡ <i>Lệnh thủ công (nếu cần gõ trong terminal Colab):</i> <code>java -Djava.awt.headless=true -jar JDownloader.jar</code>
        </div>
        """
        cred_box = widgets.VBox([
            widgets.HTML(cred_info_html),
            widgets.HBox([self.myjd_email, self.myjd_pass, self.btn_login, self.btn_view_log])
        ])
        self.cred_accordion = widgets.Accordion(children=[cred_box])
        self.cred_accordion.set_title(0, '⚙️ Cấu hình MyJDownloader (Tùy chọn)')
        self.cred_accordion.selected_index = None  # collapsed by default

        self._bind_events()

    def _get_disk_html(self) -> str:
        """Render real-time Colab local disk space indicator."""
        info = get_colab_disk_info()
        free_gb = info.get("free_gb", 0.0)
        total_gb = info.get("total_gb", 0.0)
        used_gb = info.get("used_gb", 0.0)
        pct_free = (free_gb / total_gb * 100.0) if total_gb > 0 else 0.0
        return (
            f"<div style='margin: 4px 0 8px 0; padding: 7px 12px; background-color: #172317; border: 1px solid #2e7d32; border-radius: 4px; font-size: 13px; display: flex; align-items: center; justify-content: space-between;'>"
            f"<span>💾 <b>Dung lượng Colab khả dụng:</b> <span style='color: #00e676; font-weight: bold; font-size: 14px;'>{free_gb} GB</span> / {total_gb} GB (Đã dùng: {used_gb} GB — Trống: {pct_free:.1f}%)</span>"
            f"<span style='color: #81c784; font-size: 12px;'>⚡ Stream trực tiếp vào Google Drive (Đĩa Colab không bị giảm)</span>"
            f"</div>"
        )


    def _bind_events(self):
        """Bind UI events."""
        self.btn_resolve.on_click(self._on_resolve_links)
        self.btn_quick.on_click(self._on_quick_download)
        self.btn_sample.on_click(self._on_insert_sample)
        self.btn_up.on_click(self._on_queue_up)
        self.btn_down.on_click(self._on_queue_down)
        self.btn_sort.on_click(self._on_queue_sort)
        self.btn_select_all.on_click(self._on_queue_select_all)
        self.btn_none.on_click(self._on_queue_select_none)
        self.btn_start.on_click(self._on_queue_start)
        self.btn_cancel.on_click(self._on_queue_cancel)
        self.btn_remove.on_click(self._on_queue_remove)

    def _on_insert_sample(self, b=None):
        """Insert sample Yandex Disk link into Links input."""
        current = self.text_area.value.strip()
        if not current:
            self.text_area.value = SAMPLE_DOWNLOAD_URL
        elif SAMPLE_DOWNLOAD_URL not in current:
            self.text_area.value = current + "\n" + SAMPLE_DOWNLOAD_URL
        self.status_label.value = "<span style='color:#66bb6a;'>📋 Inserted sample link: <code>" + SAMPLE_DOWNLOAD_URL + "</code></span>"

    def _on_login_myjd(self, b=None):
        """Handle MyJDownloader login, save credentials permanently to Google Drive, and start cloud daemon."""
        email = self.myjd_email.value.strip()
        pwd = self.myjd_pass.value.strip()

        if not email or not pwd:
            self.status_label.value = "<span style='color:#ff9800;'>⚠️ Vui lòng nhập đầy đủ Email và Mật khẩu tài khoản my.jdownloader.org.</span>"
            return

        self.btn_login.disabled = True
        self.btn_login.description = "Đang lưu & kết nối..."
        self.status_label.value = "<span style='color:#29b6f6;'>💾 Đang lưu cấu hình vào Google Drive & Khởi động JDownloader 2 Cloud...</span>"

        # 1. Save credentials permanently to Google Drive and local installation
        save_credentials_to_drive(email=email, password=pwd, device_name="Colab_Downloader")
        self.jd.configure_settings(email=email, password=pwd, device_name="Colab_Downloader")

        # 2. Restart/Start JDownloader 2 daemon with the new credentials
        def update_msg(msg: str):
            self.status_label.value = f"<span style='color:#29b6f6;'>{msg}</span>"

        try:
            self.jd.restart_service_with_cloud(email=email, password=pwd, device_name="Colab_Downloader", status_callback=update_msg)
        except Exception as e:
            safe_print(f"Note restarting JD2: {e}")

        # 3. Update UI immediately to success
        self.btn_login.disabled = False
        self.btn_login.description = "Đã lưu (Drive) ✅"
        self.btn_login.button_style = "success"
        self.btn_login.icon = "check"

        self.status_label.value = (
            "<div style='margin-top:6px; padding:10px 14px; background-color:#1e3b2b; border:1px solid #2e7d32; border-radius:4px; font-size:13px;'>"
            "💾 <b>Đã lưu thông tin đăng nhập vào Google Drive thành công!</b> (Vĩnh viễn không mất khi reset Colab)<br>"
            "📁 File cấu hình: <code>MyDrive/JDownloader/cfg/org.jdownloader.api.myjdownloader.MyJDownloaderSettings.json</code><br>"
            "🚀 <b>JDownloader 2 Daemon</b> đang kết nối tới <code>my.jdownloader.org</code>...<br>"
            "🌐 Thiết bị <b>Colab_Downloader</b> sẽ xuất hiện trên <a href='https://my.jdownloader.org' target='_blank' style='color:#00e676; font-weight:bold; text-decoration:underline;'>my.jdownloader.org</a> hoặc App di động sau vài giây."
            "</div>"
        )

        # 4. Background verification of device appearance on my.jdownloader.org
        def _verify_device_bg():
            time.sleep(3)
            try:
                self.jd.jd_api.connect(email, pwd)
                for _ in range(10):
                    time.sleep(2)
                    self.jd.jd_api.update_devices()
                    devs = self.jd.jd_api.list_devices() or []
                    if any(d.get("name") == "Colab_Downloader" for d in devs):
                        self.status_label.value = (
                            "<div style='margin-top:6px; padding:10px 14px; background-color:#1e3b2b; border:1px solid #2e7d32; border-radius:4px; font-size:13px;'>"
                            "✅ <b>Thiết bị 'Colab_Downloader' đã ONLINE trên my.jdownloader.org!</b><br>"
                            "🌐 Bạn đã có thể quản lý, kiểm tra hàng đợi và giải captcha từ xa trên <a href='https://my.jdownloader.org' target='_blank' style='color:#00e676; font-weight:bold; text-decoration:underline;'>my.jdownloader.org</a>."
                            "</div>"
                        )
                        break
            except Exception:
                pass

        threading.Thread(target=_verify_device_bg, daemon=True).start()

    def _on_view_log(self, b=None):
        """Display recent JDownloader 2 log lines."""
        logs = self.jd.get_service_logs(max_lines=20)
        formatted_logs = logs.replace("<", "&lt;").replace(">", "&gt;")
        self.status_label.value = (
            f"<div style='margin-top:6px; padding:8px; background-color:#111; border:1px solid #444; border-radius:4px; font-family:monospace; font-size:12px; max-height:200px; overflow-y:auto; color:#a5d6a7;'>"
            f"<b>📜 JDownloader 2 Service Log (Recent):</b><br><pre style='margin:0; white-space:pre-wrap;'>{formatted_logs}</pre>"
            f"</div>"
        )

    def _ensure_service_ready(self) -> bool:
        """Make sure JDownloader 2 is initialized and connected."""
        if self.jd._connected and self.jd.device:
            return True

        def update_msg(msg: str):
            self.status_label.value = f"<span style='color:#29b6f6;'>{msg}</span>"

        # Check Colab Drive Mount
        if IS_COLAB:
            if not os.path.exists(os.path.join(COLAB_ROOT, "drive", "MyDrive")) and \
               not os.path.exists(os.path.join(COLAB_ROOT, "drive", "My Drive")):
                update_msg("📂 Mounting Google Drive (/content/drive)...")
                try:
                    from google.colab import drive
                    drive.mount('/content/drive')
                except Exception as e:
                    print(f"Drive mount error: {e}")

        os.makedirs(DRIVE_DOWNLOADS, exist_ok=True)
        self.jd.install_prerequisites(update_msg)
        self.jd.download_jdownloader_jar(update_msg)
        self.jd.configure_settings(
            email=self.myjd_email.value.strip(),
            password=self.myjd_pass.value.strip(),
            download_folder=DRIVE_DOWNLOADS
        )
        ready = self.jd.start_service(update_msg)
        return ready

    def _extract_input_urls(self) -> List[str]:
        raw = self.text_area.value.strip()
        if not raw:
            return []
        return [line.strip() for line in raw.splitlines() if line.strip()]

    def update_queue_display(self):
        """Refresh the Queue listbox."""
        options = []
        for i, item in enumerate(self.pending_items, 1):
            name = item.get("name") or item.get("filename") or "File"
            size_str = format_size(item.get("bytesTotal", 0))
            display_text = f"{i}. 📦 {name} ({size_str}) → MyDrive/Downloads/{name}"
            options.append(display_text)

        self.queue_list.options = options
        self.queue_list.value = tuple(options)

    def _on_resolve_links(self, b=None):
        """Resolve links with JDownloader 2 Linkgrabber and populate Queue Preview."""
        urls = self._extract_input_urls()
        if not urls:
            self.status_label.value = "<span style='color:#ffa000;'>⚠️ Please paste at least one link.</span>"
            return

        self.btn_resolve.disabled = True
        self.btn_quick.disabled = True
        self.status_label.value = "<span style='color:#29b6f6;'>🔍 Initializing JDownloader 2 Engine...</span>"

        if not self._ensure_service_ready():
            self.btn_resolve.disabled = False
            self.btn_quick.disabled = False
            self.status_label.value = "<span style='color:#e53935;'>❌ Could not connect to JDownloader 2 service.</span>"
            return

        self.status_label.value = f"<span style='color:#29b6f6;'>🔍 JDownloader 2 LinkGrabber: Crawling {len(urls)} link(s)...</span>"

        try:
            # Clear old linkgrabber list
            try:
                self.jd.device.linkgrabber.clear_list()
            except Exception:
                pass

            # Add links to Linkgrabber with direct destination in Google Drive
            self.jd.device.linkgrabber.add_links([{
                "autostart": False,
                "links": "\n".join(urls),
                "packageName": "Colab_Queue",
                "destinationFolder": DRIVE_DOWNLOADS,
                "overwritePackagizerRules": True
            }])

            # Poll is_collecting until crawling finishes
            start_wait = time.time()
            time.sleep(2)
            while time.time() - start_wait < 45:
                try:
                    if not self.jd.device.linkgrabber.is_collecting():
                        break
                except Exception:
                    pass
                time.sleep(1.5)

            # Query crawled links
            crawled = self.jd.device.linkgrabber.query_links([{
                "bytesTotal": True,
                "status": True,
                "enabled": True,
                "hosts": True,
                "url": True,
                "availability": True
            }])

            self.pending_items = crawled if isinstance(crawled, list) else []

        except Exception as e:
            self.status_label.value = f"<span style='color:#e53935;'>❌ LinkGrabber error: {str(e)[:100]}</span>"
            self.btn_resolve.disabled = False
            self.btn_quick.disabled = False
            return

        self.btn_resolve.disabled = False
        self.btn_quick.disabled = False

        if not self.pending_items:
            self.status_label.value = "<span style='color:#ffa000;'>⚠️ No downloadable items resolved by JDownloader 2. Check if links are valid.</span>"
            return

        self.update_queue_display()
        self.queue_container.layout.display = 'block'
        self.status_label.value = f"<span style='color:#66bb6a;'>✅ JDownloader 2 resolved {len(self.pending_items)} item(s). Click <b>Start Download</b> to begin.</span>"

    def _on_quick_download(self, b=None):
        """Bypass Queue Preview and start downloading immediately."""
        urls = self._extract_input_urls()
        if not urls:
            self.status_label.value = "<span style='color:#ffa000;'>⚠️ Please paste at least one link.</span>"
            return

        self.queue_container.layout.display = 'none'
        self.btn_resolve.disabled = True
        self.btn_quick.disabled = True
        self.progress_bar.value = 0.0
        self.progress_bar.description = "Starting..."
        self.status_label.value = "<span style='color:#29b6f6;'>⚡ Đang khởi tạo luồng tải trực tiếp vào Google Drive...</span>"

        # Check if all links are direct/Yandex streamable for instant acceleration
        yandex_or_direct = [u for u in urls if "yandex" in u.lower() or u.lower().startswith("http")]
        if yandex_or_direct and all("yandex" in u.lower() for u in urls):
            threading.Thread(target=self._stream_direct_worker, args=(urls,), daemon=True).start()
            return

        if not self._ensure_service_ready():
            self.btn_resolve.disabled = False
            self.btn_quick.disabled = False
            self.status_label.value = "<span style='color:#e53935;'>❌ Could not connect to JDownloader 2 service.</span>"
            return

        try:
            # Clear old queue
            try:
                self.jd.device.linkgrabber.clear_list()
            except Exception:
                pass

            # Add with autostart and direct destination in Google Drive
            self.status_label.value = "<span style='color:#29b6f6;'>⚡ JDownloader 2: Đang gửi link vào hàng đợi tải Google Drive...</span>"
            self.jd.device.linkgrabber.add_links([{
                "autostart": True,
                "links": "\n".join(urls),
                "packageName": "Colab_Quick",
                "destinationFolder": DRIVE_DOWNLOADS,
                "overwritePackagizerRules": True
            }])

            # Trigger download start
            time.sleep(1)
            try:
                self.jd.device.downloadcontroller.start_downloads()
            except Exception:
                pass

            # Monitor downloads and drive moves in background thread
            self._start_download_monitor()

        except Exception as e:
            self.status_label.value = f"<span style='color:#e53935;'>❌ Quick Download error: {str(e)[:100]}</span>"
            self.btn_resolve.disabled = False
            self.btn_quick.disabled = False

    def _stream_direct_worker(self, urls: List[str]):
        """High-speed direct streaming worker bypassing JD2 startup delay for supported hosts."""
        self.is_downloading = True
        completed = []
        try:
            for idx, url in enumerate(urls, 1):
                self.status_label.value = f"<span style='color:#29b6f6;'>🔍 Đang giải mã liên kết tải nhanh [{idx}/{len(urls)}]: <code>{url[:60]}...</code></span>"
                yd_info = resolve_yandex_disk_direct_link(url) if "yandex" in url.lower() else None

                if not yd_info:
                    # Fallback to JD2
                    if self._ensure_service_ready():
                        self.jd.device.linkgrabber.add_links([{"autostart": True, "links": url, "destinationFolder": DRIVE_DOWNLOADS}])
                        self.jd.device.downloadcontroller.start_downloads()
                    continue

                direct_url, filename, filesize = yd_info
                safe_name = sanitize_filename(filename)
                dest_path = os.path.join(DRIVE_DOWNLOADS, safe_name)
                total_size_str = format_size(filesize)

                self.status_label.value = f"⬇️ Đang stream trực tiếp vào Google Drive: <b>{safe_name}</b> ({total_size_str})"
                start_t = time.time()
                bytes_done = 0
                last_t = start_t
                last_bytes = 0

                with requests.get(direct_url, stream=True, timeout=30) as resp:
                    resp.raise_for_status()
                    total_len = int(resp.headers.get('content-length', filesize))

                    with open(dest_path, "wb") as f_out:
                        for chunk in resp.iter_content(chunk_size=4 * 1024 * 1024):
                            if not chunk or not self.is_downloading:
                                break
                            f_out.write(chunk)
                            bytes_done += len(chunk)
                            curr_t = time.time()

                            if curr_t - last_t >= 1.0:
                                speed_bps = (bytes_done - last_bytes) / (curr_t - last_t)
                                speed_str = f"{format_size(int(speed_bps))}/s"
                                pct = (bytes_done / total_len * 100.0) if total_len > 0 else 0.0
                                self.progress_bar.value = max(0.0, min(100.0, pct))
                                self.progress_bar.description = f"Drive: {int(pct)}% ({speed_str})"
                                self.disk_info_widget.value = self._get_disk_html()
                                cur_disk = get_colab_disk_info()
                                self.status_label.value = (
                                    f"⬇️ Streaming trực tiếp vào Google Drive: <b>{safe_name}</b> ({format_size(bytes_done)} / {total_size_str}) "
                                    f"— <code>{speed_str}</code> (Colab trống: <b>{cur_disk['free_gb']} GB</b> - Không giảm!)"
                                )
                                last_t = curr_t
                                last_bytes = bytes_done

                completed.append(safe_name)

        except Exception as e:
            self.status_label.value = f"<span style='color:#e53935;'>⚠️ Streaming error: {str(e)[:100]}</span>"
        finally:
            self.is_downloading = False
            self.btn_resolve.disabled = False
            self.btn_quick.disabled = False
            self.progress_bar.value = 100.0
            self.progress_bar.description = "Done"
            self.disk_info_widget.value = self._get_disk_html()
            final_disk = get_colab_disk_info()
            self.status_label.value = (
                f"<div style='margin-top:8px; padding:10px; background-color:#2e3b2e; border-radius:4px;'>"
                f"<b>🎉 Hoàn thành tải {len(completed)} file trực tiếp vào Google Drive!</b><br>"
                f"📂 Thư mục lưu: <code>MyDrive/Downloads/</code><br>"
                f"<small style='color:#a5d6a7;'>⚡ Đĩa Colab cục bộ khả dụng: <b>{final_disk['free_gb']} GB</b> (Giữ nguyên 100% không giảm).</small>"
                f"</div>"
            )

    def _scan_and_transfer_completed(self, completed_files: set):
        """Scan local temporary directories for any downloaded files and transfer them to Google Drive."""
        temp_dirs = [
            os.path.join(COLAB_ROOT, "downloads"),
            os.path.join(self.jd.jd_dir, "downloads") if hasattr(self.jd, "jd_dir") else None,
            "./downloads"
        ]
        for t_dir in temp_dirs:
            if not t_dir or not os.path.exists(t_dir):
                continue
            for entry in os.listdir(t_dir):
                f_path = os.path.join(t_dir, entry)
                if os.path.isfile(f_path) and not entry.endswith(".part"):
                    success, dest = transfer_to_google_drive(f_path, DRIVE_DOWNLOADS)
                    if success:
                        completed_files.add(entry)
                        safe_print(f"📦 Transferred local fallback file to Drive: {entry}")

    def _kick_jd2_downloads(self, link_ids=None, package_ids=None):
        """Force JDownloader 2 to start downloading and actively recover from STOPPED state."""
        try:
            self.jd.device.downloadcontroller.start_downloads()
        except Exception:
            pass
        try:
            self.jd.device.downloadcontroller.force_download(link_ids or [], package_ids or [])
        except Exception:
            pass
        try:
            self.jd.device.downloads.force_download(link_ids or [], package_ids or [])
        except Exception:
            pass

    def _start_download_monitor(self, link_ids=None, package_ids=None):
        """Dispatch monitor loop via asyncio if running on IPython/Colab event loop, else via daemon thread."""
        loop = None
        try:
            import asyncio
            try:
                loop = asyncio.get_running_loop()
            except RuntimeError:
                loop = asyncio.get_event_loop()
        except Exception:
            loop = None

        if loop and loop.is_running():
            loop.create_task(self._monitor_and_transfer_async(link_ids, package_ids))
        else:
            threading.Thread(target=self._monitor_and_transfer, args=(link_ids, package_ids), daemon=True).start()

    def _on_queue_start(self, b=None):
        """Move selected items from Linkgrabber to download list and start downloading."""
        selected_strings = list(self.queue_list.value)
        if not selected_strings:
            self.status_label.value = "<span style='color:#ffa000;'>⚠️ Please select at least one item from Queue.</span>"
            return

        selected_indices = [int(s.split('.')[0].strip()) - 1 for s in selected_strings]
        selected_tasks = [self.pending_items[i] for i in selected_indices if 0 <= i < len(self.pending_items)]

        link_ids = [item.get("uuid") for item in selected_tasks if item.get("uuid")]
        package_ids = list(set([item.get("packageUUID") for item in selected_tasks if item.get("packageUUID")]))

        self.queue_container.layout.display = 'none'
        self.btn_resolve.disabled = True
        self.btn_quick.disabled = True
        self.progress_bar.value = 0.0
        self.progress_bar.description = "Starting..."
        self.status_label.value = "<span style='color:#29b6f6;'>▶️ Đang chuyển tác vụ vào Danh sách tải JDownloader 2 & Khởi động luồng tải...</span>"

        # Check if items contain direct Yandex links for fast streaming
        item_urls = [t.get("url") for t in selected_tasks if t.get("url")]
        if not item_urls:
            raw_urls = self._extract_input_urls()
            if raw_urls and all("yandex" in u.lower() for u in raw_urls):
                item_urls = raw_urls

        if item_urls and all("yandex" in u.lower() for u in item_urls):
            threading.Thread(target=self._stream_direct_worker, args=(item_urls,), daemon=True).start()
            return

        try:
            # 1. Move items to download list in JDownloader 2
            try:
                self.jd.device.linkgrabber.move_to_downloadlist(link_ids, package_ids)
            except Exception as move_err:
                safe_print(f"Linkgrabber move note: {move_err}")
                try:
                    self.jd.device.linkgrabber.move_to_downloadlist(link_ids, [])
                except Exception:
                    pass

            # 2. Kick download controller to start downloading
            self._kick_jd2_downloads(link_ids, package_ids)

            # 3. Monitor downloads and drive moves
            self._start_download_monitor(link_ids=link_ids, package_ids=package_ids)

        except Exception as e:
            self.status_label.value = f"<span style='color:#e53935;'>❌ Start Download error: {str(e)[:100]}</span>"
            self.btn_resolve.disabled = False
            self.btn_quick.disabled = False

    def _check_drive_target_files(self) -> List[str]:
        """Check Google Drive downloads folder for newly created/extracted files."""
        found = []
        if os.path.exists(DRIVE_DOWNLOADS):
            try:
                for item in os.listdir(DRIVE_DOWNLOADS):
                    p = os.path.join(DRIVE_DOWNLOADS, item)
                    if os.path.isfile(p) and not item.endswith(".part"):
                        found.append(item)
                    elif os.path.isdir(p) and not item.startswith("."):
                        found.append(item + "/")
            except Exception:
                pass
        return found

    def _monitor_tick(self, link_ids, package_ids, completed_files, wait_cycles) -> Tuple[bool, int]:
        """
        Perform a single monitoring and status refresh iteration.
        Handles downloading, auto-extraction, package completion, and Drive persistence.
        Returns (is_complete, wait_cycles).
        """
        try:
            # Check download controller state: auto-kick if STOPPED
            try:
                ctrl_state = self.jd.device.downloadcontroller.get_current_state()
            except Exception:
                ctrl_state = "UNKNOWN"

            # Query download links with default params (includes extractionStatus, name, bytes)
            try:
                links = self.jd.device.downloads.query_links()
            except Exception:
                links = []

            # Query download packages
            try:
                packages = self.jd.device.downloads.query_packages()
            except Exception:
                packages = []

            # Check if any archive is currently extracting or completed extraction
            is_extracting = False
            extraction_done = False
            for l in (links or []):
                stat = (l.get("status") or "").lower()
                ext_stat = (l.get("extractionStatus") or "").lower()
                if "extracting" in stat or "extracting" in ext_stat:
                    is_extracting = True
                if "extraction ok" in stat or "extraction: ok" in stat or "finished" in ext_stat or "success" in ext_stat:
                    extraction_done = True

            # If controller dropped to STOPPED with pending links, kick it back to RUNNING
            if links and ctrl_state == "STOPPED" and not is_extracting:
                unfinished = [l for l in links if not l.get("finished")]
                if unfinished:
                    self._kick_jd2_downloads(link_ids, package_ids)
                    try:
                        ctrl_state = self.jd.device.downloadcontroller.get_current_state()
                    except Exception:
                        pass

            speed_bps = 0
            try:
                speed_bps = self.jd.device.downloadcontroller.get_speed_in_bytes() or 0
            except Exception:
                pass
            if speed_bps == 0 and links:
                speed_bps = sum(l.get("speed", 0) or 0 for l in links)
            speed_str = f"{format_size(speed_bps)}/s"

            # Track newly finished links into completed_files
            for link in (links or []):
                fname = link.get("name")
                is_fin = link.get("finished") or extraction_done
                if is_fin and fname and fname not in completed_files:
                    completed_files.add(fname)
                    safe_print(f"✅ Tải & xử lý hoàn tất trong Google Drive: {fname}")

            # Check files present in Google Drive destination
            drive_files = self._check_drive_target_files()
            for df in drive_files:
                completed_files.add(df)

            self._scan_and_transfer_completed(completed_files)
            cur_disk = get_colab_disk_info()
            self.disk_info_widget.value = self._get_disk_html()

            # --- CASE 1: No active links found ---
            if not links:
                # If packages exist and are finished, OR files exist in Drive and we waited past startup
                all_pkgs_done = packages and all(p.get("finished", False) for p in packages)
                if (all_pkgs_done or len(completed_files) > 0) and wait_cycles >= 2:
                    safe_print("🎉 Tất cả gói tải và giải nén đã hoàn tất trong Google Drive!")
                    return True, wait_cycles

                self.status_label.value = (
                    f"<span style='color:#29b6f6;'>⏳ Đang kết nối hoster & đồng bộ Danh sách tải JDownloader 2 "
                    f"(chờ {wait_cycles*2}s)... (Colab trống: <b>{cur_disk['free_gb']} GB</b>)</span>"
                )
                self.progress_bar.description = f"Connecting... ({wait_cycles*2}s)"
                self._update_log_output(f"Đang đồng bộ hoster... (chờ {wait_cycles*2}s)")
                return False, wait_cycles

            # --- CASE 2: Links present ---
            total_bytes = sum(l.get("bytesTotal", 0) or 0 for l in links)
            loaded_bytes = sum(l.get("bytesLoaded", 0) or 0 for l in links)
            all_finished = all(l.get("finished", False) for l in links) or (packages and all(p.get("finished", False) for p in packages))

            pct = (loaded_bytes / total_bytes * 100.0) if total_bytes > 0 else 0.0
            if all_finished or extraction_done:
                pct = 100.0

            running_links = [l for l in links if l.get("running")]
            active_links = [l for l in links if not l.get("finished")]

            if is_extracting:
                curr_file = links[0].get("name") or "Archive"
                self.progress_bar.value = 99.0
                self.progress_bar.description = "Extracting..."
                status_text = (
                    f"📦 <b>[JDownloader 2] Đang tự động giải nén:</b> <code>{curr_file}</code> trực tiếp trong Google Drive... "
                    f"(Colab trống: <b>{cur_disk['free_gb']} GB</b>)"
                )
            elif running_links:
                curr_file = running_links[0].get("name") or "File"
                curr_status = running_links[0].get("status") or "Downloading"
                self.progress_bar.value = max(0.0, min(100.0, pct))
                self.progress_bar.description = f"Drive: {int(pct)}% ({speed_str})"
                status_text = (
                    f"⬇️ Streaming trực tiếp vào Google Drive: <b>{curr_file}</b> "
                    f"[{curr_status}] — <code>{speed_str}</code> (Colab trống: <b>{cur_disk['free_gb']} GB</b> - Không giảm!)"
                )
            elif active_links:
                curr_file = active_links[0].get("name") or "File"
                curr_status = active_links[0].get("status") or "Connecting..."
                self.progress_bar.value = max(0.0, min(100.0, pct))
                self.progress_bar.description = f"Drive: {int(pct)}% ({curr_status})"
                status_text = (
                    f"⏳ [JDownloader 2: {ctrl_state}] <b>{curr_file}</b>: {curr_status} "
                    f"— Đang duy trì kết nối... (Colab trống: <b>{cur_disk['free_gb']} GB</b>)"
                )
            else:
                self.progress_bar.value = 100.0
                self.progress_bar.description = "Done"
                status_text = f"⏳ Đang hoàn tất gói tải vào Google Drive... (Colab trống: <b>{cur_disk['free_gb']} GB</b>)"

            self.status_label.value = status_text
            self._update_log_output(f"[{int(pct)}%] {speed_str} | {status_text}")

            # Check overall completion
            if (all_finished or extraction_done) and (len(completed_files) >= len(links) or len(drive_files) > 0):
                return True, wait_cycles

        except Exception as loop_ex:
            self.status_label.value = f"<span style='color:#ffa000;'>⏳ Đang đồng bộ trạng thái: {str(loop_ex)[:60]}...</span>"
            self._update_log_output(f"Đang đồng bộ: {str(loop_ex)[:60]}")

        return False, wait_cycles

    def _update_log_output(self, msg: str):
        """Print clean progress update to standard output for live feedback."""
        clean = re.sub(r'<[^>]+>', '', msg).strip()
        if clean:
            safe_print(f"⚡ [JD2] {clean}")

    def _monitor_and_transfer(self, link_ids=None, package_ids=None):
        """
        Periodically check download progress (threaded runner).
        Files are streamed directly into Google Drive (DRIVE_DOWNLOADS) with 0 Colab disk space usage.
        """
        self.is_downloading = True
        completed_files = set()
        wait_cycles = 0

        # Wait briefly for items to land in download list, then kick start
        time.sleep(1.5)
        self._kick_jd2_downloads(link_ids, package_ids)

        try:
            while self.is_downloading:
                time.sleep(1.5)
                wait_cycles += 1
                done, wait_cycles = self._monitor_tick(link_ids, package_ids, completed_files, wait_cycles)
                if done:
                    break

        finally:
            self._scan_and_transfer_completed(completed_files)
            self.is_downloading = False
            self.btn_resolve.disabled = False
            self.btn_quick.disabled = False
            self.progress_bar.value = 100.0
            self.progress_bar.description = "Done"
            self.disk_info_widget.value = self._get_disk_html()
            final_disk = get_colab_disk_info()
            summary = (
                f"<div style='margin-top:8px; padding:10px; background-color:#2e3b2e; border-radius:4px;'>"
                f"<b>🎉 Hoàn thành tải {len(completed_files)} file trực tiếp vào Google Drive!</b><br>"
                f"📂 Thư mục: <code>MyDrive/Downloads/</code><br>"
                f"<small style='color:#a5d6a7;'>⚡ Đĩa Colab cục bộ khả dụng: <b>{final_disk['free_gb']} GB</b> (Giữ nguyên 100% không giảm).</small>"
                f"</div>"
            )
            self.status_label.value = summary

    async def _monitor_and_transfer_async(self, link_ids=None, package_ids=None):
        """
        Periodically check download progress (async runner for Colab/IPython).
        Yields control to the main event loop so that ipywidgets comm flushes to browser in real-time.
        """
        self.is_downloading = True
        completed_files = set()
        wait_cycles = 0

        await asyncio.sleep(1.5)
        self._kick_jd2_downloads(link_ids, package_ids)

        try:
            while self.is_downloading:
                await asyncio.sleep(1.5)
                wait_cycles += 1
                done, wait_cycles = self._monitor_tick(link_ids, package_ids, completed_files, wait_cycles)
                if done:
                    break

        finally:
            self._scan_and_transfer_completed(completed_files)
            self.is_downloading = False
            self.btn_resolve.disabled = False
            self.btn_quick.disabled = False
            self.progress_bar.value = 100.0
            self.progress_bar.description = "Done"
            self.disk_info_widget.value = self._get_disk_html()
            final_disk = get_colab_disk_info()
            summary = (
                f"<div style='margin-top:8px; padding:10px; background-color:#2e3b2e; border-radius:4px;'>"
                f"<b>🎉 Hoàn thành tải {len(completed_files)} file trực tiếp vào Google Drive!</b><br>"
                f"📂 Thư mục: <code>MyDrive/Downloads/</code><br>"
                f"<small style='color:#a5d6a7;'>⚡ Đĩa Colab cục bộ khả dụng: <b>{final_disk['free_gb']} GB</b> (Giữ nguyên 100% không giảm).</small>"
                f"</div>"
            )
            self.status_label.value = summary

    def _on_queue_up(self, b=None):
        selected = list(self.queue_list.value)
        if not selected:
            return
        indices = sorted([int(s.split('.')[0]) - 1 for s in selected])
        for idx in indices:
            if idx > 0 and idx - 1 not in indices:
                self.pending_items[idx], self.pending_items[idx - 1] = self.pending_items[idx - 1], self.pending_items[idx]
        self.update_queue_display()
        new_sel = [self.queue_list.options[max(0, i - 1)] for i in indices]
        self.queue_list.value = tuple(new_sel)

    def _on_queue_down(self, b=None):
        selected = list(self.queue_list.value)
        if not selected:
            return
        indices = sorted([int(s.split('.')[0]) - 1 for s in selected], reverse=True)
        for idx in indices:
            if idx < len(self.pending_items) - 1 and idx + 1 not in indices:
                self.pending_items[idx], self.pending_items[idx + 1] = self.pending_items[idx + 1], self.pending_items[idx]
        self.update_queue_display()
        new_sel = [self.queue_list.options[min(len(self.pending_items) - 1, i + 1)] for i in indices]
        self.queue_list.value = tuple(new_sel)

    def _on_queue_sort(self, b=None):
        self.pending_items.sort(key=lambda t: (t.get("name") or "").lower(), reverse=not self.sort_ascending)
        self.sort_ascending = not self.sort_ascending
        self.btn_sort.description = "Sort Z-A" if not self.sort_ascending else "Sort A-Z"
        self.update_queue_display()

    def _on_queue_select_all(self, b=None):
        self.queue_list.value = tuple(self.queue_list.options)

    def _on_queue_select_none(self, b=None):
        self.queue_list.value = ()

    def _on_queue_cancel(self, b=None):
        self.queue_container.layout.display = 'none'
        self.status_label.value = "<span style='color:#aaa;'>Queue preview closed.</span>"

    def _on_queue_remove(self, b=None):
        selected = list(self.queue_list.value)
        if not selected:
            return
        indices_to_remove = {int(s.split('.')[0]) - 1 for s in selected}
        removed_items = [self.pending_items[i] for i in indices_to_remove]
        self.pending_items = [t for i, t in enumerate(self.pending_items) if i not in indices_to_remove]

        # Remove from JD Linkgrabber if possible
        try:
            link_ids = [item.get("uuid") for item in removed_items if item.get("uuid")]
            package_ids = list(set([item.get("packageUUID") for item in removed_items if item.get("packageUUID")]))
            if link_ids or package_ids:
                self.jd.device.linkgrabber.remove_links(link_ids, package_ids)
        except Exception:
            pass

        self.update_queue_display()
        if not self.pending_items:
            self.queue_container.layout.display = 'none'
            self.status_label.value = "<span style='color:#aaa;'>All items removed from queue.</span>"

    def render(self):
        """Render the complete widget tree."""
        button_row = widgets.HBox([
            self.btn_resolve,
            self.btn_quick,
            self.btn_sample
        ], layout=widgets.Layout(margin='6px 0'))

        app_view = widgets.VBox([
            self.css_styles,
            self.disk_info_widget,
            self.cred_accordion,
            self.text_area,
            button_row,
            self.queue_container,
            self.progress_bar,
            self.status_label
        ], layout=widgets.Layout(width='100%', padding='10px'))

        try:
            display(app_view)
        except Exception:
            safe_print("ℹ️ Note: ipywidgets display failed in current environment. Use download_direct(urls) to run directly.")


# --- DIRECT DRIVE STREAMING ENGINE (BYPASS 87GB COLAB DISK) ---
def resolve_yandex_disk_direct_link(yandex_url: str) -> Optional[Tuple[str, str, int]]:

    """
    Resolve public Yandex Disk URL to direct HTTPS CDN streaming link and metadata.
    Returns (download_url, filename, size_bytes) or None.
    """
    try:
        api_meta = f"https://cloud-api.yandex.net/v1/disk/public/resources?public_key={yandex_url.strip()}"
        res_meta = requests.get(api_meta, timeout=15)
        if res_meta.status_code != 200:
            return None
        data_meta = res_meta.json()
        filename = data_meta.get("name", "download.bin")
        filesize = data_meta.get("size", 0)

        api_dl = f"https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key={yandex_url.strip()}"
        res_dl = requests.get(api_dl, timeout=15)
        if res_dl.status_code != 200:
            return None
        download_url = res_dl.json().get("href")
        if download_url:
            return (download_url, filename, filesize)
    except Exception:
        pass
    return None


def download_direct(
    urls: str | List[str],
    download_folder: Optional[str] = None,
    status_callback: Optional[callable] = None,
    max_duration_seconds: Optional[int] = None
) -> Dict[str, Any]:
    """
    Direct-to-Drive streaming downloader engine.
    Bypasses Colab's ~87GB virtual disk limit by writing directly into mounted Google Drive.
    Works seamlessly in Antigravity IDE without requiring ipywidgets or unpkg.com CDN.

    :param urls: URL string (newline-separated) or list of URLs.
    :param download_folder: Destination folder (defaults to DRIVE_DOWNLOADS: /content/drive/MyDrive/Downloads).
    :param status_callback: Optional function to receive status messages.
    :param max_duration_seconds: Optional timeout (useful for verification runs).
    """
    if isinstance(urls, str):
        url_list = [u.strip() for u in urls.strip().splitlines() if u.strip()]
    else:
        url_list = [str(u).strip() for u in urls if str(u).strip()]

    if not url_list:
        msg = "⚠️ No URLs provided for direct download."
        safe_print(msg)
        return {"success": False, "error": msg, "files": []}

    target_folder = download_folder or DRIVE_DOWNLOADS
    os.makedirs(target_folder, exist_ok=True)

    def log_status(msg: str):
        if status_callback:
            status_callback(msg)
        safe_print(msg)

    initial_disk = get_colab_disk_info()
    log_status("=" * 70)
    log_status("🚀 CLOUD-TO-DRIVE DIRECT STREAMING ENGINE")
    log_status(f"🎯 Target Destination: {target_folder}")
    log_status(f"💾 Colab Local Disk Status: {initial_disk['used_gb']} GB used / {initial_disk['total_gb']} GB total ({initial_disk['free_gb']} GB free)")
    log_status("⚡ Direct FUSE Streaming Enabled: Local disk usage remains 0% during download.")
    log_status("=" * 70)

    downloaded_files = []

    # Process each URL
    for idx, url in enumerate(url_list, 1):
        log_status(f"\n[{idx}/{len(url_list)}] Processing link: {url}")

        # Check if Yandex Disk link
        yd_info = None
        if "yandex" in url.lower():
            log_status("🔍 Resolving Yandex Disk link via public API...")
            yd_info = resolve_yandex_disk_direct_link(url)

        if yd_info:
            direct_url, filename, filesize = yd_info
            safe_filename = sanitize_filename(filename)
            dest_file = os.path.join(target_folder, safe_filename)
            size_str = format_size(filesize)
            log_status(f"📦 File: {safe_filename} ({size_str})")
            log_status(f"⬇️ Streaming directly into Google Drive: {dest_file}")

            start_t = time.time()
            bytes_transferred = 0
            last_report_t = start_t
            last_bytes = 0

            try:
                with requests.get(direct_url, stream=True, timeout=30) as resp:
                    resp.raise_for_status()
                    total_len = int(resp.headers.get('content-length', filesize))

                    with open(dest_file, "wb") as f_out:
                        for chunk in resp.iter_content(chunk_size=4 * 1024 * 1024):
                            if not chunk:
                                break
                            f_out.write(chunk)
                            bytes_transferred += len(chunk)
                            curr_t = time.time()

                            # Periodic progress log every 2 seconds
                            if curr_t - last_report_t >= 2.0:
                                speed_bps = (bytes_transferred - last_bytes) / (curr_t - last_report_t)
                                speed_str = f"{format_size(int(speed_bps))}/s"
                                pct = (bytes_transferred / total_len * 100.0) if total_len > 0 else 0.0
                                cur_disk = get_colab_disk_info()
                                safe_print(
                                    f"   ⏳ Streamed: {format_size(bytes_transferred)}/{format_size(total_len)} "
                                    f"({pct:.1f}%) | Speed: {speed_str} | "
                                    f"Colab Disk: {cur_disk['free_gb']} GB Free (Unchanged!)",
                                    end="\r"
                                )
                                last_report_t = curr_t
                                last_bytes = bytes_transferred


                            if max_duration_seconds and (curr_t - start_t) >= max_duration_seconds:
                                log_status(f"\n⏱️ Reached max duration limit of {max_duration_seconds}s for verification.")
                                break

                final_disk = get_colab_disk_info()
                log_status(f"\n✅ Streamed successfully to Drive: {dest_file}")
                log_status(f"💾 Colab Local Disk: {final_disk['free_gb']} GB free (Local disk space preserved!)")
                downloaded_files.append({"name": safe_filename, "path": dest_file, "size": bytes_transferred})
                continue

            except Exception as e:
                log_status(f"⚠️ Direct HTTP stream error: {e}. Falling back to JDownloader 2 engine...")

        # JDownloader 2 Engine fallback / multi-host support
        jd_service = JD2Service()
        jd_service.configure_settings(download_folder=target_folder)
        log_status("🚀 Starting JDownloader 2 Headless Service...")
        try:
            jd_service.start_service(log_status)
            if jd_service._connected and jd_service.device:
                log_status(f"➕ Adding URL to JDownloader 2 LinkGrabber: {url}")
                jd_service.device.linkgrabber.add_links([{
                    "autostart": True,
                    "links": url,
                    "packageName": "Drive_Streaming_Queue",
                    "destinationFolder": target_folder,
                    "overwritePackagizerRules": True
                }])
                log_status("⚡ JDownloader 2 is streaming directly to Google Drive...")
                downloaded_files.append({"url": url, "status": "added_to_jd2"})
        except Exception as ex:
            log_status(f"⚠️ JDownloader 2 execution note: {ex}")

    log_status("\n" + "=" * 70)
    log_status("🎉 DIRECT STREAMING COMPLETE!")
    log_status(f"📂 Check files in Google Drive: {target_folder}")
    log_status("=" * 70)

    return {
        "success": True,
        "files": downloaded_files,
        "initial_disk": initial_disk,
        "final_disk": get_colab_disk_info()
    }


# --- MAIN RUNNER ---
def run_app():
    """Khởi chạy ứng dụng Colab Downloader (chỉ hiển thị 1 ô giao diện duy nhất)."""
    clear_output(wait=True)
    safe_print("🚀 CloudToDrive v2.1 — JDownloader 2 Cloud Manager đã sẵn sàng!")
    app = ColabDownloaderApp()
    app.render()
    return None


if __name__ == "__main__":
    run_app()
